# 01 - Clean the data and build the modelling table

Reads the Zenodo dataset with PySpark, cleans it, and writes
`curated/features.parquet` and publishes the Parquet parts to the bucket
(`models/features/`), the project's distributed object store. Run order: 01 -> 03 -> 02.
Re-download the data with `docker compose run --rm ingest`.


In [ ]:
import json
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

JOBS = os.environ.get('JOBS_DIR', '/home/jovyan/jobs')
if JOBS not in sys.path:
    sys.path.insert(0, JOBS)
from schemas import FPS_COLS, GAMES_SCHEMA, GPUS_SCHEMA, REFERENCE_GPUS

from spark_session import get_spark

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
spark = get_spark('gpu-recsys-features')
print('spark', spark.version, '| master', spark.sparkContext.master,
      '| driver heap MiB', spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() // 1048576)

In [ ]:
prov = json.loads((DATA / '_provenance.json').read_text())
print(f"version {prov['version']}  doi {prov['doi']}  license {prov['license']}")

games = (
    spark.read.csv(str(DATA / 'games.csv'), header=True, schema=GAMES_SCHEMA)
    .dropDuplicates(['name', 'release_year'])
)
gpus = spark.read.csv(str(DATA / 'gpus.csv'), header=True, schema=GPUS_SCHEMA)
print(f'games {games.count():,}  gpus {gpus.count():,}')

## Parse the hardware text

The `min_*` / `rec_*` hardware fields are free text. They are parsed with Spark SQL
functions only (no Python UDFs). When a row lists alternatives ("A or B") the weaker
option is the binding requirement, so numbers are combined with `array_min`.
Units are normalised (MB to GB, MHz to GHz).


In [ ]:
def _low(col):
    return F.regexp_replace(F.lower(col), r'(\d),(\d)', r'$1.$2')


def _all(col, pattern, idx=1):
    return F.regexp_extract_all(col, F.lit(pattern), idx)


def _gpu(col):
    low = _low(col)
    gb = _all(low, r'(\d+(?:\.\d+)?)\s*gb')
    mb = _all(low, r'(\d+(?:\.\d+)?)\s*mb')
    vram = F.concat(
        F.transform(gb, lambda x: F.when(x.cast('double') >= 128, x.cast('double') / 1024).otherwise(x.cast('double'))),
        F.transform(mb, lambda x: x.cast('double') / 1024),
    )
    dx = F.transform(_all(low, r'(?:directx|dx)\s*(\d+)'), lambda x: x.cast('int'))
    return F.struct(
        F.coalesce(
            F.when(low.rlike(r'nvidia|geforce|gtx|rtx|\bgts\b|\bgt\b'), 'nvidia'),
            F.when(low.rlike(r'\bamd\b|radeon|\brx\b|hd (?!graphics)\d|\bati\b'), 'amd'),
            F.when(low.rlike(r'intel|iris|uhd|hd graphics'), 'intel'),
        ).alias('vendor'),
        F.array_min(vram).alias('vram_gb'),
        F.array_min(dx).alias('api'),
        low.rlike(r'iris|uhd graphics|hd graphics|onboard|integrated|radeon \d+m\b').alias('integrated'),
    )


def _cpu(col):
    low = _low(col)
    ghz = F.transform(
        _all(low, r'(\d+(?:\.\d+)?)\s*ghz'),
        lambda x: F.when(x.cast('double') >= 100, x.cast('double') / 1000).otherwise(x.cast('double')),
    )
    mhz = F.transform(_all(low, r'(\d+(?:\.\d+)?)\s*mhz'), lambda x: x.cast('double') / 1000)
    mult = F.transform(
        F.filter(_all(low, r'(\d+)\s*\*\s*(\d+)', 2), lambda x: x.cast('double') > 500),
        lambda x: x.cast('double') / 1000,
    )
    cores = F.filter(F.concat(
        F.transform(_all(low, r'(\d+)\s*[- ]?core'), lambda x: x.cast('double')),
        F.transform(_all(low, r'\bx([2468])\b'), lambda x: x.cast('double')),
        F.transform(_all(low, r'\((\d+)\s*cpus?\)'), lambda x: x.cast('double')),
        F.array(
            F.when(low.rlike('dual'), 2.0),
            F.when(low.rlike('quad'), 4.0),
            F.when(low.rlike('hexa'), 6.0),
            F.when(low.rlike('octa'), 8.0),
        ),
    ), lambda x: x.isNotNull())
    return F.struct(
        F.coalesce(
            F.when(low.rlike(r'intel|\bi[3579]\b|core i|pentium|celeron'), 'intel'),
            F.when(low.rlike(r'\bamd\b|ryzen|athlon|\bfx\b|fx-'), 'amd'),
        ).alias('vendor'),
        F.array_min(F.concat(ghz, mhz, mult)).alias('ghz'),
        F.array_min(cores).cast('int').alias('cores'),
    )


games = (
    games
    .withColumn('mg', _gpu('min_gpu')).withColumn('rg', _gpu('rec_gpu'))
    .withColumn('mc', _cpu('min_cpu')).withColumn('rc', _cpu('rec_cpu'))
    .select(
        '*',
        F.col('mg.vendor').alias('min_gpu_vendor'),
        F.col('mg.vram_gb').alias('min_vram_gb'),
        F.col('mg.api').alias('min_gpu_api'),
        F.col('mg.integrated').alias('min_gpu_integrated'),
        F.col('rg.vendor').alias('rec_gpu_vendor'),
        F.col('rg.vram_gb').alias('rec_vram_gb'),
        F.col('rg.api').alias('rec_gpu_api'),
        F.col('rg.integrated').alias('rec_gpu_integrated'),
        F.col('mc.vendor').alias('min_cpu_vendor'),
        F.col('mc.ghz').alias('min_ghz'),
        F.col('mc.cores').alias('min_cores'),
        F.col('rc.vendor').alias('rec_cpu_vendor'),
        F.col('rc.ghz').alias('rec_ghz'),
        F.col('rc.cores').alias('rec_cores'),
    )
    .drop('mg', 'rg', 'mc', 'rc')
)
print('min_vram_gb populated:', games.filter('min_vram_gb is not null').count(),
      '| min_ghz populated:', games.filter('min_ghz is not null').count())

## Clean units, invalid values and caps

- RAM >= 128 is really MB, so it is divided by 1024; 0 becomes null.
- `is_capped` marks rows at the 360 FPS reporting ceiling.
- Engine-capped rows are dropped: they report the same FPS on every GPU.


In [ ]:
def ram_gb(name):
    col = F.col(name).cast('double')
    return F.when(col <= 0, None).when(col >= 128, col / 1024).otherwise(col)


games = (
    games
    .withColumn('min_ram_gb', ram_gb('min_ram_gb'))
    .withColumn('rec_ram_gb', ram_gb('rec_ram_gb'))
    .withColumn('is_capped', (F.greatest(*FPS_COLS) >= 360).cast('int'))
)
print('engine-capped rows:', games.filter('engine_fps_cap is not null').count(),
      '| hard-capped (360):', games.filter('is_capped = 1').count())

## Target: the lightest GPU that reaches 60 FPS

The 12 FPS columns are unpivoted and joined to benchmark scores; the lowest score
that clears 60 FPS becomes `required_g3d`.


In [ ]:
arms = [part for key, col in zip(REFERENCE_GPUS, FPS_COLS) for part in (f"'{key}'", f'`{col}`')]
long_df = (
    games
    .select('name', 'release_year', F.expr(f'stack({len(FPS_COLS)}, {", ".join(arms)}) as (gpu_key, fps)'))
    .filter(F.col('fps').isNotNull())
)

gpu_scores = {row['name']: row['g3d_mark'] for row in gpus.select('name', 'g3d_mark').collect()}
score = F.create_map(*[F.lit(v) for key, name in REFERENCE_GPUS.items() for v in (key, gpu_scores[name])])

target = (
    long_df
    .withColumn('g3d_mark', score[F.col('gpu_key')])
    .filter(F.col('fps') >= 60)
    .groupBy('name', 'release_year')
    .agg(F.min('g3d_mark').alias('required_g3d'))
)
print(f'games with a resolvable target: {target.count():,}')

## What the curated table keeps

Raw `min_*` / `rec_*` columns stay for later tasks. `min_gpu_g3d` / `rec_gpu_g3d`
are inputs to the published FPS columns, so they are not used as features.
Features: `game_age`, `min_ram_gb`, `min_vram_gb`, `min_ghz`, `is_capped`.


In [ ]:
curated = (
    games
    .join(target, on=['name', 'release_year'], how='inner')
    .filter(F.col('engine_fps_cap').isNull())
    .withColumn('game_age', F.lit(2026) - F.col('release_year'))
    .withColumn('min_cores', F.coalesce('min_cores', F.lit(0)))
    .withColumn('rec_cores', F.coalesce('rec_cores', F.lit(0)))
    .na.fill({'min_ram_gb': 0.0, 'rec_ram_gb': 0.0, 'min_vram_gb': 0.0, 'min_ghz': 0.0})
)
curated.cache()
print(f'curated rows: {curated.count():,}')

In [ ]:
OUT = DATA / 'curated'
curated.repartition(2).write.mode('overwrite').parquet(str(OUT / 'features.parquet'))

print('wrote', OUT / 'features.parquet')

from storage import get_bucket, model_bucket_name, upload_file

bucket = get_bucket()
parts = sorted((OUT / 'features.parquet').glob('part-*.parquet'))
for part in parts:
    upload_file(bucket, f'models/features/{part.name}', part, 'application/octet-stream')
print(f'uploaded {len(parts)} Parquet parts to gs://{model_bucket_name()}/models/features/')